# Forecast Diurnal Cycle Analysis
CESM2 hindcast ensemble — diurnal cycle amplitude and phase by forecast lead day

- Set `experiment` to switch between **CESM2-CAM6v2** (original 6-hourly hindcasts) and **CESM2-ERA5init** (ERA5-initialized 3-hourly hindcasts)
- Set `var_name` to analyze any available CAM variable (PRECT, TMQ, PSL, TS, UBOT, …)
- Composites are averaged over all selected start dates × ensemble members
- Separate diurnal cycle is computed for each lead day (1, 2, 3, …)
- Output: Evans maps (phase=hue, amplitude=saturation) and regional line plots per lead day

In [28]:
import importlib
import sys
import os

sys.path.insert(0, '/glade/work/rneale/git/python-scripts/diurnal_cycle/forecast')
sys.path.insert(0, '/glade/work/rneale/git/python-scripts/diurnal_cycle')

import forecast_diurnal_utils as fcutils
import diurnal_cycle_utils    as dcutils
importlib.reload(fcutils)
importlib.reload(dcutils)

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

print('Imports OK')

Imports OK


In [29]:
from dask.distributed import Client
from dask_jobqueue import PBSCluster

In [30]:
cluster = PBSCluster(
    account="P03010039",
    interface="ext",
    walltime="12:00:00",
    queue="main",
    cores=1,
    memory="8GB",
    processes=4,
    local_directory="/glade/derecho/scratch/rneale/dask-temp",
    log_directory="/glade/derecho/scratch/rneale/dask-logs"
)

cluster.scale(jobs=8)
client = Client(cluster)

client

/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45315 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: /node/crhtc45.hpc.ucar.edu/34172/proxy/45315/status,
Dashboard: /node/crhtc45.hpc.ucar.edu/34172/proxy/45315/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.95:39581,Workers: 0
Dashboard: /node/crhtc45.hpc.ucar.edu/34172/proxy/45315/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [31]:
# ============================================================
# SETTINGS
# ============================================================

# --- Experiment selection ---
# 'CESM2-CAM6v2'   : original CESM2 CAM6v2 S2S hindcasts   (6-hourly, up to 21 members)
# 'CESM2-ERA5init' : ERA5-initialized CESM2 hindcasts        (3-hourly, up to 11 members)
experiment = 'CESM2-ERA5init'

# --- Variable ---
# CESM2-CAM6v2   : PRECT, TMQ, U10, PSL
# CESM2-ERA5init : PRECT, TMQ, PSL, TS, UBOT, QBOT
var_name = 'QBOT'

# --- Season / year range ---
season     = 'JJA'
year_start = 1999
year_end   = 2021

# --- Ensemble members (None = all available for the chosen experiment) ---
ens_members = [1,2,3,4]

# --- Lead days and layout ---
nlead_days = 10
n_cols     = 5      # n_cols × 2 rows of panels

# --- Map domain ---
domain = 'CONUS'    # 'GLOBAL' | 'TROPICS' | 'US' | 'CONUS'

_DOMAINS = {
    'GLOBAL':  {'lat_range': (-40.,  40.), 'lon_range': (-180., 180.)},
    'TROPICS': {'lat_range': (-20.,  20.), 'lon_range': (-180., 180.)},
    'US':      {'lat_range': ( 15.,  60.), 'lon_range': (-140.,  -55.)},
    'CONUS':   {'lat_range': ( 22.,  52.), 'lon_range': (-130.,  -65.)},
}
if domain not in _DOMAINS:
    raise ValueError(f"Unknown domain '{domain}'. Choose: {list(_DOMAINS)}")
lat_range   = _DOMAINS[domain]['lat_range']
lon_range   = _DOMAINS[domain]['lon_range']
show_states = domain in ('US', 'CONUS')

# --- Harmonic analysis ---
n_harm       = 2
harm_names   = ['Diurnal (H1, 24 h)', 'Semi-diurnal (H2, 12 h)']
harm_periods = [24.0, 12.0]
harm_fnames  = ['H1_diurnal', 'H2_semidiurnal']

# --- Evans amplitude range (None = auto-derived from data after loading) ---
min_amp_list   = [None, None]   # one per harmonic; set e.g. [0.1, 0.05] to fix
max_amp_list   = [None, None]
hue_offset     = 0.5
discrete_wheel = True
min_amp_raw    = None           # raw diurnal max — None = auto
max_amp_raw    = None

# --- Regional line plots ---
regions_line = {
    'Midwest':        (35., 45., -100.,  -90., None),
    'Southeast US':   (25., 35.,  -92.,  -78., 'land'),
    'SE Coast Ocean': (30., 35.,  -80.,  -70., 'ocean'),
}

# --- Output directory ---
dir_fig = '/glade/u/home/rneale/python/python-figs/diurnal_cycle/forecast/'
os.makedirs(dir_fig, exist_ok=True)

In [32]:
# ── Resolve experiment configuration ─────────────────────────────────────────
importlib.reload(fcutils)

if experiment not in fcutils.EXPERIMENTS:
    raise ValueError(f"Unknown experiment {experiment!r}. "
                     f"Choose: {list(fcutils.EXPERIMENTS)}")

exp_cfg   = fcutils.EXPERIMENTS[experiment]
data_dir  = exp_cfg['data_dir']
dt_hours  = exp_cfg['dt_hours']
exp_label = exp_cfg['label']
exp_tag   = experiment.replace('CESM2-', '')   # 'CAM6v2' | 'ERA5init' for filenames

if ens_members is None:
    ens_members = list(range(exp_cfg['n_ens']))

vset      = fcutils.VAR_SETTINGS.get(var_name, {})
var_label = vset.get('long_name', var_name)
var_units = vset.get('units', '')
var_cmap  = vset.get('cmap', None)

# Fixed contour levels for PRECT on CONUS/US; auto-derived otherwise
var_levels = ([0, 0.25, 0.5, 1, 1.5, 2, 3, 4, 5, 6, 8, 10]
              if var_name == 'PRECT' and domain in ('US', 'CONUS') else None)
# Precipitation levels always available for IMERG comparison
prcp_levels = ([0, 0.25, 0.5, 1, 1.5, 2, 3, 4, 5, 6, 8, 10]
               if domain in ('US', 'CONUS') else None)

print(f'Experiment : {experiment}')
print(f'             {exp_label}')
print(f'Variable   : {var_name}  —  {var_label}  [{var_units}]')
print(f'Season     : {season}  {year_start}–{year_end}')
print(f'dt_hours   : {dt_hours}')
print(f'Ensemble   : {len(ens_members)} members  ({ens_members[0]}–{ens_members[-1]})')
print(f'Lead days  : 1–{nlead_days}  ({n_cols} cols × 2 rows)')
print(f'Domain     : {domain}  states={show_states}')

Experiment : CESM2-ERA5init
             CESM2 ERA5-initialized hindcasts (3-hourly)
Variable   : QBOT  —  Lowest Model Level Water Vapour  [g/kg]
Season     : JJA  1999–2021
dt_hours   : 3
Ensemble   : 4 members  (1–4)
Lead days  : 1–10  (5 cols × 2 rows)
Domain     : CONUS  states=True


In [33]:
importlib.reload(fcutils)

start_dates = fcutils.get_start_dates_for(
    experiment,
    data_dir   = data_dir,
    season     = season,
    year_start = year_start,
    year_end   = year_end,
)

print(f'Start dates found: {len(start_dates)}')
print(f'  First: {start_dates[0]}   Last: {start_dates[-1]}')
print(f'Total files to load: {len(start_dates) * len(ens_members)}')

Start dates found: 302
  First: 1999-06-07   Last: 2021-08-30
Total files to load: 1208


In [ ]:
importlib.reload(fcutils)

print(f'Loading {var_name}: {len(start_dates)} start dates × '
      f'{len(ens_members)} members → {nlead_days} lead days ...')

dc_leads = fcutils.load_experiment(
    experiment  = experiment,
    var_name    = var_name,
    data_dir    = data_dir,
    start_dates = start_dates,
    ens_members = ens_members,
    nlead_days  = nlead_days,
    dt_hours    = dt_hours,
)

lat = dc_leads['lat'].values
lon = dc_leads['lon'].values

print(f'\ndc_leads shape: {dc_leads.shape}')
print(f'  dims: {dict(dc_leads.sizes)}')
print(f'  lat:  {lat[0]:.2f} to {lat[-1]:.2f}')
print(f'  lon:  {lon[0]:.2f} to {lon[-1]:.2f}')
print(f'  time-of-day (UTC h): {dc_leads["time_of_day"].values}')

for d in range(nlead_days):
    m = float(np.nanmean(dc_leads.isel(lead_day=d)))
    print(f'  Lead day {d+1}: mean {var_name} = {m:.4g} {var_units}')

# ── Auto-derive plot ranges from the loaded data ──────────────────────────────
_ranges = fcutils.auto_plot_ranges(dc_leads)
if var_levels is None:
    var_levels = _ranges['levels']
min_amp_list = [(_ranges['min_amp'] if v is None else v) for v in min_amp_list]
max_amp_list = [(_ranges['max_amp'] if v is None else v) for v in max_amp_list]
if min_amp_raw is None:
    min_amp_raw = _ranges['min_amp']
if max_amp_raw is None:
    max_amp_raw = _ranges['max_amp']

print(f'\nPlot ranges for {var_name}:')
print(f'  Mean-map levels : {[f"{v:.4g}" for v in var_levels]}')
print(f'  Evans H1 amp    : {min_amp_list[0]:.4g} – {max_amp_list[0]:.4g} {var_units}')
print(f'  Evans H2 amp    : {min_amp_list[1]:.4g} – {max_amp_list[1]:.4g} {var_units}')

Loading QBOT: 302 start dates × 4 members → 10 lead days ...
  100/1208  loaded=100  missing=0  no_var=0
  200/1208  loaded=200  missing=0  no_var=0
  300/1208  loaded=300  missing=0  no_var=0
  400/1208  loaded=400  missing=0  no_var=0
  500/1208  loaded=500  missing=0  no_var=0


In [ ]:
importlib.reload(fcutils)

# ============================================================
# HARMONIC ANALYSIS — all lead days
# ============================================================

amplitude, phase_utc, var_exp, mean_field = fcutils.compute_lead_harmonics(
    dc_leads, n_harm=n_harm, dt_hours=dt_hours)

# amplitude  : (nlead, n_harm, nlat, nlon)
# phase_utc  : (nlead, n_harm, nlat, nlon)
# var_exp    : (nlead, n_harm, nlat, nlon)
# mean_field : (nlead, nlat, nlon)  — time-mean value at each grid point

# Convert UTC phase → local solar time for each lead day and harmonic
phase_lst = np.zeros_like(phase_utc)
for d in range(nlead_days):
    for ih in range(n_harm):
        phase_lst[d, ih] = fcutils.phase_utc_to_lst(
            phase_utc[d, ih], lon, harm_periods[ih])

# Raw diurnal max (lead-day composite)
amp_raw_list       = []
phase_lst_raw_list = []
for d in range(nlead_days):
    a_raw, ph_utc_raw = fcutils.compute_raw_diurnal(dc_leads.isel(lead_day=d),
                                                     dt_hours=dt_hours)
    amp_raw_list.append(a_raw)
    phase_lst_raw_list.append(fcutils.phase_utc_to_lst(ph_utc_raw, lon, 24.0))

amp_raw_leads       = np.stack(amp_raw_list)        # (nlead, nlat, nlon)
phase_lst_raw_leads = np.stack(phase_lst_raw_list)

print('Harmonic analysis complete.')
for d in range(nlead_days):
    for ih in range(n_harm):
        print(f'  Lead day {d+1}  H{ih+1}: '
              f'amp_max={np.nanmax(amplitude[d,ih]):.4g} {var_units}  '
              f'var_exp_mean={np.nanmean(var_exp[d,ih]):.1%}')

In [ ]:
# ============================================================
# FIGURE 0: Region box map
# ============================================================

importlib.reload(fcutils)

fig = fcutils.plot_region_boxes(
    regions     = regions_line,
    lat_range   = (20., 55.),
    lon_range   = (-130., -60.),
    show_states = True,
    title       = f'Averaging regions — {list(regions_line.keys())}',
)
fname = os.path.join(dir_fig,
    f'forecast_regions_{"-".join(k.replace(" ","_") for k in regions_line)}.png')
fig.savefig(fname, dpi=150, bbox_inches='tight')
print(f'Saved: {fname}')
plt.show()
plt.close(fig)

# ============================================================
# FIGURE 1: Mean field panel (lead days 1–nlead_days)
# ============================================================

fig = fcutils.plot_forecast_mean_panel(
    mean_field, lat, lon,
    lead_days   = dc_leads['lead_day'].values,
    lat_range   = lat_range,
    lon_range   = lon_range,
    show_states = show_states,
    n_cols      = n_cols,
    levels      = var_levels,
    cmap        = var_cmap,
    units       = f'{var_label} ({var_units})',
    title       = (f'{exp_tag}  {season} {year_start}–{year_end}  '
                   f'ens={len(ens_members)}  {var_label} — time mean'),
)
fname = os.path.join(dir_fig,
    f'forecast_{exp_tag}_{domain}_{season}_{year_start}-{year_end}_{var_name}_mean.png')
fig.savefig(fname, dpi=150, bbox_inches='tight')
print(f'Saved: {fname}')
plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# FIGURE 2+: Evans Phase/Amplitude Panel — one per harmonic
# ============================================================

for ih in range(n_harm):
    fig = fcutils.plot_forecast_panel(
        amplitude      = amplitude[:, ih],
        phase_lst      = phase_lst[:, ih],
        lat=lat, lon=lon,
        lead_days      = dc_leads['lead_day'].values,
        min_amp        = min_amp_list[ih],
        max_amp        = max_amp_list[ih],
        period_hours   = harm_periods[ih],
        hue_offset     = hue_offset,
        lat_range      = lat_range,
        lon_range      = lon_range,
        discrete_wheel = discrete_wheel,
        dt_hours       = dt_hours,
        show_states    = show_states,
        n_cols         = n_cols,
        title          = (f'{exp_tag}  {season} {year_start}–{year_end}  '
                          f'ens={len(ens_members)}  {var_label}  {harm_names[ih]}  '
                          f'(phase=hue, amp [{var_units}]=saturation)'),
    )
    fname = os.path.join(dir_fig,
        f'forecast_{exp_tag}_{domain}_{season}_{year_start}-{year_end}'
        f'_{var_name}_evans_{harm_fnames[ih]}.png')
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    print(f'Saved: {fname}')
    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# FIGURE 3: Raw Diurnal Max Panel
# ============================================================

fig = fcutils.plot_forecast_panel(
    amplitude      = amp_raw_leads,
    phase_lst      = phase_lst_raw_leads,
    lat=lat, lon=lon,
    lead_days      = dc_leads['lead_day'].values,
    min_amp        = min_amp_raw,
    max_amp        = max_amp_raw,
    period_hours   = 24.0,
    hue_offset     = hue_offset,
    lat_range      = lat_range,
    lon_range      = lon_range,
    discrete_wheel = discrete_wheel,
    dt_hours       = dt_hours,
    show_states    = show_states,
    n_cols         = n_cols,
    title          = (f'{exp_tag}  {season} {year_start}–{year_end}  '
                      f'ens={len(ens_members)}  {var_label}  Raw diurnal max  '
                      f'(phase=hue, amplitude=saturation)'),
)
fname = os.path.join(dir_fig,
    f'forecast_{exp_tag}_{domain}_{season}_{year_start}-{year_end}'
    f'_{var_name}_evans_raw_max.png')
fig.savefig(fname, dpi=150, bbox_inches='tight')
print(f'Saved: {fname}')
plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# FIGURE 4: Regional Diurnal Cycle Line Plots
# One PNG per region saved to dir_fig
# ============================================================

importlib.reload(fcutils)

figs = fcutils.plot_regional_diurnal_lines(
    dc_leads,
    lat=lat, lon=lon,
    regions      = regions_line,
    dt_hours     = dt_hours,
    ylabel       = f'{var_label} ({var_units})',
    title_prefix = (f'{exp_tag}  {season} {year_start}–{year_end}  '
                    f'ens={len(ens_members)}  {var_name}'),
    dir_fig      = dir_fig,
    fname_prefix = (f'forecast_{exp_tag}_{domain}_{season}'
                    f'_{year_start}-{year_end}_{var_name}_regional'),
)

for fig in figs:
    plt.show(fig)
    plt.close(fig)

---
## IMERG Validation
Observed GPM IMERG V07B 3-hourly 1-degree — same season and year range as the forecasts

In [ ]:
# ============================================================
# LOAD IMERG + COMPUTE HARMONICS
# ============================================================

importlib.reload(fcutils)

dc_imerg = fcutils.load_imerg_diurnal(
    imerg_dir  = fcutils.IMERG_DIR,
    season     = season,
    year_start = year_start,
    year_end   = year_end,
)

lat_i = dc_imerg['lat'].values
lon_i = dc_imerg['lon'].values

print(f'dc_imerg shape : {dc_imerg.shape}')
print(f'  lat  : {lat_i[0]:.1f} to {lat_i[-1]:.1f}')
print(f'  lon  : {lon_i[0]:.1f} to {lon_i[-1]:.1f}')
print(f'  time-of-day (UTC h): {dc_imerg["time_of_day"].values}')
for d in range(dc_imerg.sizes['time_of_day']):
    m = float(np.nanmean(dc_imerg.isel(time_of_day=d)))
    print(f'  TOD {dc_imerg["time_of_day"].values[d]:2.0f}h : mean = {m:.4f} mm/day')

# Harmonics (dt_hours=3 for 3-hourly IMERG)
amp_i, ph_utc_i, var_exp_i, mean_i = dcutils.compute_harmonics(
    dc_imerg, n_harm=n_harm, dt_hours=3)

# UTC → LST
ph_lst_i = np.zeros_like(ph_utc_i)
for ih in range(n_harm):
    ph_lst_i[ih] = fcutils.phase_utc_to_lst(ph_utc_i[ih], lon_i, harm_periods[ih])

# Raw diurnal max
amp_raw_i, ph_utc_raw_i = dcutils.compute_raw_diurnal(dc_imerg, dt_hours=3)
ph_lst_raw_i = fcutils.phase_utc_to_lst(ph_utc_raw_i, lon_i, 24.0)

print('\nIMERG harmonic summary:')
for ih in range(n_harm):
    print(f'  H{ih+1}: amp_max={np.nanmax(amp_i[ih]):.3f} mm/day  '
          f'var_exp={np.nanmean(var_exp_i[ih]):.1%}')
print(f'  Overall mean precip: {np.nanmean(mean_i):.4f} mm/day')


In [ ]:
# ============================================================
# FIGURE I1: IMERG mean precipitation (validation)
# ============================================================

import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(
    111, projection=ccrs.PlateCarree(central_longitude=180))

cf = dcutils.plot_mean_precip(
    ax, mean_i, lat_i, lon_i,
    title   = (f'IMERG  {season} {year_start}–{year_end}  '
               f'Mean Precipitation (observed)'),
    levels  = prcp_levels,
    cmap    = dcutils.CMAP_PRCP,
    lat_range  = lat_range,
    lon_range  = lon_range,
    show_states = show_states,
)

cbar = fig.colorbar(cf, ax=ax, orientation='horizontal',
                    fraction=0.04, pad=0.08, shrink=0.85)
cbar.set_label('Precipitation (mm/day)', fontsize=9)
cbar.ax.tick_params(labelsize=8)
fig.tight_layout()

fname = os.path.join(dir_fig,
    f'imerg_{domain}_{season}_{year_start}-{year_end}_mean_precip.png')
fig.savefig(fname, dpi=150, bbox_inches='tight')
print(f'Saved: {fname}')
plt.show()
plt.close(fig)


In [ ]:
# ============================================================
# FIGURE I2: IMERG diurnal cycle phase/amplitude — Evans maps
# One panel per harmonic, shared color wheel
# ============================================================

for ih in range(n_harm):
    fig = plt.figure(figsize=(10, 5))
    gs  = gridspec.GridSpec(
        1, 2, width_ratios=[4, 0.8],
        left=0.04, right=0.96, bottom=0.08, top=0.88,
        wspace=0.08,
    )
    ax_map   = fig.add_subplot(
        gs[0, 0], projection=ccrs.PlateCarree(central_longitude=180))
    ax_wheel = fig.add_subplot(gs[0, 1], projection='polar')

    dcutils.plot_evans_map(
        ax_map, fig,
        phase_lst    = ph_lst_i[ih],
        amplitude    = amp_i[ih],
        lat=lat_i, lon=lon_i,
        min_amp      = min_amp_list[ih],
        max_amp      = max_amp_list[ih],
        period_hours = harm_periods[ih],
        title        = (f'IMERG  {season} {year_start}–{year_end}  '
                        f'{harm_names[ih]}  (observed)'),
        hue_offset   = hue_offset,
        lat_range    = lat_range,
        lon_range    = lon_range,
        ax_wheel     = ax_wheel,
        discrete_wheel = discrete_wheel,
        dt_hours     = 3,
        show_states  = show_states,
    )

    fname = os.path.join(dir_fig,
        f'imerg_{domain}_{season}_{year_start}-{year_end}'
        f'_evans_{harm_fnames[ih]}.png')
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    print(f'Saved: {fname}')
    plt.show()
    plt.close(fig)

# Raw diurnal max (peak-to-trough / 2, no harmonic fitting)
fig = plt.figure(figsize=(10, 5))
gs  = gridspec.GridSpec(
    1, 2, width_ratios=[4, 0.8],
    left=0.04, right=0.96, bottom=0.08, top=0.88, wspace=0.08)
ax_map   = fig.add_subplot(
    gs[0, 0], projection=ccrs.PlateCarree(central_longitude=180))
ax_wheel = fig.add_subplot(gs[0, 1], projection='polar')

dcutils.plot_evans_map(
    ax_map, fig,
    phase_lst    = ph_lst_raw_i,
    amplitude    = amp_raw_i,
    lat=lat_i, lon=lon_i,
    min_amp      = min_amp_raw,
    max_amp      = max_amp_raw,
    period_hours = 24.0,
    title        = (f'IMERG  {season} {year_start}–{year_end}  '
                    f'Raw diurnal max  (observed)'),
    hue_offset   = hue_offset,
    lat_range    = lat_range,
    lon_range    = lon_range,
    ax_wheel     = ax_wheel,
    discrete_wheel = discrete_wheel,
    dt_hours     = 3,
    show_states  = show_states,
)

fname = os.path.join(dir_fig,
    f'imerg_{domain}_{season}_{year_start}-{year_end}_evans_raw_max.png')
fig.savefig(fname, dpi=150, bbox_inches='tight')
print(f'Saved: {fname}')
plt.show()
plt.close(fig)
